# 06 Firm characteristics and AI-announcer classification

Firm descriptives (Chapter 3) and the AI-announcer classification (Chapter 4).

Universe: every S&P 500 and STOXX Europe 600 member between 2018 and 2026, joined to Compustat by
gvkey from the ISIN or SEDOL match of the constituent file (inner join, no imputation of missing accounting
data). One row per company (gvkey). $Adopt_i = 1$ if any of the company's securities appears in the 685-event
candidate roster, before the membership and earnings filters (Section 4.1).
Features come from the last fiscal year ending on or before 2017-12-31, before any announcement.

In [1]:
import warnings

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from src.common import input_path, TABLES, INTERIM, GICS_TO_BROAD, load_constituents, load_returns

warnings.filterwarnings("ignore", message="divide by zero encountered in log")

us_c, eu_c = load_constituents()
comp_us = pd.read_parquet(input_path("accounting_us"))
comp_eu = pd.read_parquet(input_path("accounting_global"))
assert "at_usd" in comp_eu.columns, "EU total assets must be converted to USD"


def to_sector(code):
    key = str(code).strip() if code is not None and not (isinstance(code, float) and np.isnan(code)) else ""
    return GICS_TO_BROAD.get(key[:2], "Other") if key not in ("", "None", "nan") else "Other"


def active(c):
    c = c.copy()
    c["membership_start"] = pd.to_datetime(c["membership_start"], errors="coerce")
    return c[(c["membership_start"] <= "2026-12-31") & (c["membership_end"] >= "2018-01-01")]


us_u = active(us_c).drop_duplicates(subset="permno").copy()
eu_u = active(eu_c).dropna(subset=["isin"]).drop_duplicates(subset="infocode").copy()

## Firm descriptives

Latest Compustat fiscal year per firm. EU log assets use total assets converted to USD. The ratios use
local-currency values. Table 3.3 reports market capitalisation for US firms only.

In [2]:
us_gics = us_u.dropna(subset=["permno"]).set_index(us_u["permno"].dropna().astype(int))["gics_sector_code"].to_dict()
eu_gics = (eu_u.dropna(subset=["gvkey"]).assign(gvkey=lambda d: d["gvkey"].astype(str))
           .drop_duplicates(subset="gvkey").set_index("gvkey")["gics_sector_code"].to_dict())


def latest(comp):
    comp = comp.copy()
    comp["datadate"] = pd.to_datetime(comp["datadate"])
    return comp.sort_values("datadate").groupby("gvkey").last().reset_index()


d_us = latest(comp_us)
d_us["permno"] = pd.to_numeric(d_us["permno"], errors="coerce")
d_us = d_us[d_us["permno"].isin(set(us_u["permno"].dropna().astype(int)))].copy()
for c in ["at", "xrd", "dltt", "dlc", "csho", "prcc_f"]:
    d_us[c] = pd.to_numeric(d_us[c], errors="coerce")
d_us = d_us[d_us["at"] > 0].copy()
d_us["log_assets"] = np.log(d_us["at"])
d_us["mktcap_bn"] = d_us["csho"] * d_us["prcc_f"] / 1000
d_us["sector"] = d_us["permno"].astype("Int64").map(lambda p: us_gics.get(int(p)) if pd.notna(p) else None).apply(to_sector)

d_eu = comp_eu.copy()
for c in ["at", "xrd", "dltt", "dlc", "at_usd"]:
    d_eu[c] = pd.to_numeric(d_eu[c], errors="coerce")
d_eu = latest(d_eu)
d_eu["gvkey"] = d_eu["gvkey"].astype(str)
d_eu = d_eu[d_eu["gvkey"].isin(set(eu_u["gvkey"].dropna().astype(str)))].copy()
d_eu = d_eu[(d_eu["at"] > 0) & (d_eu["at_usd"] > 0)].copy()
d_eu["log_assets"] = np.log(d_eu["at_usd"])
d_eu["mktcap_bn"] = np.nan
d_eu["sector"] = d_eu["gvkey"].map(eu_gics).apply(to_sector)

for d in (d_us, d_eu):
    d["rd_intensity"] = d["xrd"].fillna(0) / d["at"]
    d["leverage"] = (d["dltt"].fillna(0) + d["dlc"].fillna(0)) / d["at"]


def describe(d, region, sector=None):
    out = {"region": region, "sector": sector, "N_firms": len(d)}
    for col, name, nd in [("log_assets", "log_assets", 3), ("rd_intensity", "rd_intensity", 4),
                          ("leverage", "leverage", 4), ("mktcap_bn", "mktcap_bn", 2)]:
        v = d[col].dropna()
        v = v[v > 0] if col == "mktcap_bn" else v
        out[f"mean_{name}"] = v.mean() if len(v) else np.nan
        out[f"median_{name}"] = v.median() if len(v) else np.nan
    return out


rows = [describe(d_us, "US"), describe(d_eu, "EU")]
for region, d in [("US", d_us), ("EU", d_eu)]:
    rows += [describe(g, region, s) for s, g in d.groupby("sector") if len(g) >= 5]
desc = pd.DataFrame(rows)[["region", "sector", "N_firms", "mean_log_assets", "median_log_assets",
                           "mean_rd_intensity", "median_rd_intensity", "mean_leverage", "median_leverage",
                           "mean_mktcap_bn", "median_mktcap_bn"]]
desc.to_csv(TABLES / "firm_descriptives.csv", index=False)
desc[desc["sector"].isna()]

,region,sector,N_firms,mean_log_assets,median_log_assets,mean_rd_intensity,median_rd_intensity,mean_leverage,median_leverage,mean_mktcap_bn,median_mktcap_bn
0,US,None,657,10.1850,10.0610,0.0224,0.0000,0.3406,0.3238,96.7013,27.9435
1,EU,None,651,9.1752,9.0789,0.0236,0.0046,0.2838,0.2703,NaN,NaN


## Classification sample

Features: log assets, R&D / assets, missing-R&D indicator, leverage, sales / assets, log market cap,
sector dummies (reference Industrials). Ratios winsorised at the 1st/99th percentile.
EU market capitalisation comes from Datastream at the FY2017 fiscal year-end, because Datastream's coverage
is more complete than Compustat Global's (Section 3.3).

In [3]:
CUTOFF = pd.Timestamp("2017-12-31")
link = pd.read_csv(INTERIM / "event_linkage.csv")
link = link[link["identifier"].notna()]
adopters = {r: set(link.loc[link["Region"] == r, "identifier"].astype(int)) for r in ("US", "EU")}


def winsorize(s):
    return s.clip(lower=s.quantile(0.01), upper=s.quantile(0.99))


def features(chars, shares, price, level="at"):
    s = chars[chars["datadate"] <= CUTOFF].sort_values(["gvkey", "fyear"]).drop_duplicates("gvkey", keep="last").copy()
    at = s["at"].where(s["at"] > 0)
    lvl = s[level].where(s[level] > 0)
    s["log_assets"] = np.log(lvl.clip(lower=1e-6))
    s["rd_missing"] = s["xrd"].isna().astype(int)
    s["rd_intensity"] = winsorize(s["xrd"].fillna(0) / at)
    s["leverage"] = winsorize((s["dltt"].fillna(0) + s["dlc"].fillna(0)) / at)
    s["sales_to_assets"] = winsorize(s["sale"] / at)
    mc = s[shares] * s[price]
    s["log_mktcap"] = np.log(mc.where(mc > 0).clip(lower=1e-6))
    return s[["gvkey", "log_assets", "rd_intensity", "rd_missing", "leverage", "sales_to_assets",
              "log_mktcap", "sich", "datadate"]].reset_index(drop=True)


def by_company(c, sec_id):
    """One row per gvkey, keeping the still-active security; all security ids retained for the label."""
    ids = c.groupby("gvkey")[sec_id].apply(lambda s: sorted(set(s.dropna().astype(int).tolist())))
    out = c.sort_values(["gvkey", "still_active"], ascending=[True, False]).drop_duplicates("gvkey").copy()
    out["all_sec_ids"] = out["gvkey"].map(ids)
    return out


def with_sector_dummies(d):
    dummies = pd.get_dummies(d["sector"], prefix="sec").drop(columns=["sec_Industrials"])
    return pd.concat([d, dummies], axis=1)


BASE_FEATURES = ["log_assets", "rd_intensity", "rd_missing", "leverage", "sales_to_assets", "log_mktcap"]

# US
u = us_u[us_u["sector_available"]].copy()
u["permno"] = u["permno"].astype("Int64")
u = u[u["gvkey"].notna()].assign(gvkey=lambda d: d["gvkey"].astype(str))
comp_us["gvkey"] = comp_us["gvkey"].astype(str)
comp_us["datadate"] = pd.to_datetime(comp_us["datadate"])
us_df = by_company(u, "permno")[["gvkey", "all_sec_ids", "gics_sector_code"]].merge(
    features(comp_us, "csho", "prcc_f"), on="gvkey", how="inner")
us_df["sector"] = us_df["gics_sector_code"].apply(to_sector)
us_df["adopt"] = us_df["all_sec_ids"].apply(lambda ids: int(any(i in adopters["US"] for i in ids)))
us_df = with_sector_dummies(us_df)

# EU
e = eu_u[eu_u["sector_available"]].copy()
e["infocode"] = e["infocode"].astype(float).astype("Int64")
e["gvkey"] = e["gvkey"].astype(str)
eu_company = by_company(e, "infocode")
# log_mktcap from Compustat Global is replaced below by the Datastream value (Section 3.3)
comp_eu["gvkey"] = comp_eu["gvkey"].astype(str)
comp_eu["datadate"] = pd.to_datetime(comp_eu["datadate"])
eu_df = eu_company[["infocode", "all_sec_ids", "isin", "gvkey", "company_name", "gics_sector_code"]].merge(
    features(comp_eu, "cshr", "prc", level="at_usd"), on="gvkey", how="inner")
eu_df["sector"] = eu_df["gics_sector_code"].apply(to_sector)
_, eu_r = load_returns()
mc = eu_r.dropna(subset=["mktcap_usd"])[["infocode", "date", "mktcap_usd"]].sort_values("date")
eu_df["infocode"] = eu_df["infocode"].astype("Int64")
eu_df = pd.merge_asof(eu_df.sort_values("datadate"), mc.astype({"infocode": "Int64"}), left_on="datadate",
                      right_on="date", by="infocode", direction="nearest", tolerance=pd.Timedelta(days=5))
eu_df["log_mktcap"] = np.log(eu_df["mktcap_usd"].where(eu_df["mktcap_usd"] > 0))
eu_df = eu_df.drop(columns=["date", "mktcap_usd"])
eu_df["adopt"] = eu_df["all_sec_ids"].apply(lambda ids: int(any(i in adopters["EU"] for i in ids)))
eu_df = with_sector_dummies(eu_df)

for r, d in [("US", us_df), ("EU", eu_df)]:
    print(f"{r}: {len(d)} firms, {d['adopt'].sum()} adopters ({100 * d['adopt'].mean():.1f}%)")

US: 624 firms, 135 adopters (21.6%)
EU: 619 firms, 40 adopters (6.5%)


## Models

Logistic regression, decision tree and random forest, estimated separately per region.
Repeated stratified 5-fold CV (10 repeats) for ROC-AUC, F1 and PR-AUC. Confusion matrices use
out-of-fold predictions. Feature ranking: random-forest permutation importance on held-out folds.

In [4]:
MODELS = {
    "LogisticRegression": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                                    ("clf", LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000,
                                                               random_state=42))]),
    "DecisionTree": Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("clf", DecisionTreeClassifier(max_depth=5, min_samples_leaf=5,
                                                             class_weight="balanced", random_state=42))]),
    "RandomForest": Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("clf", RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=3,
                                                             class_weight="balanced", random_state=42, n_jobs=-1))]),
}


def evaluate(d):
    cols = BASE_FEATURES + [c for c in d.columns if c.startswith("sec_")]
    X, y = d[cols].astype(float).values, d["adopt"].values
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
    cv1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = {}
    for name, pipe in MODELS.items():
        s = cross_validate(clone(pipe), X, y, cv=cv, scoring=["roc_auc", "f1", "average_precision"])
        scores[name] = {k: s[f"test_{m}"] for k, m in
                        [("auc", "roc_auc"), ("f1", "f1"), ("pr_auc", "average_precision")]}
    oof = cross_val_predict(clone(MODELS["RandomForest"]), X, y, cv=cv1, method="predict")
    imp = []
    for tr, te in cv1.split(X, y):
        m = clone(MODELS["RandomForest"]).fit(X[tr], y[tr])
        imp.append(permutation_importance(m, X[te], y[te], n_repeats=10, random_state=42,
                                          scoring="roc_auc", n_jobs=-1).importances_mean)
    importance = pd.DataFrame({"feature": cols, "importance": np.mean(imp, axis=0)}).sort_values(
        "importance", ascending=False)
    return scores, confusion_matrix(y, oof), importance


rows = []
for region, d in [("US", us_df), ("Europe", eu_df)]:
    scores, cm, importance = evaluate(d)
    importance.to_csv(TABLES / f"feature_importance_{'us' if region == 'US' else 'eu'}.csv", index=False)
    for name, s in scores.items():
        rows.append(dict(model=name, region=region,
                         **{f"{k}_{stat}": getattr(v, stat)() for k, v in s.items()
                            for stat in ("mean", "std")}))
    print(region, "random forest, out-of-fold confusion matrix:\n", cm)
    print(importance.head(8).to_string(index=False))
ml = pd.DataFrame(rows)[["model", "region", "auc_mean", "auc_std", "f1_mean", "f1_std", "pr_auc_mean", "pr_auc_std"]]
ml.to_csv(TABLES / "classification_cv.csv", index=False)
ml

US random forest, out-of-fold confusion matrix:
 [[437  52]
 [ 65  70]]
       feature  importance
    log_mktcap      0.0735
sec_Technology      0.0531
  rd_intensity      0.0441
    log_assets      0.0082
sec_Financials      0.0068
    sec_Energy      0.0058
  sec_Consumer      0.0045
 sec_Utilities      0.0042


Europe random forest, out-of-fold confusion matrix:
 [[563  16]
 [ 31   9]]
           feature  importance
        log_mktcap      0.0751
          leverage      0.0383
      rd_intensity      0.0296
        log_assets      0.0167
   sales_to_assets      0.0110
        rd_missing      0.0093
      sec_Consumer      0.0046
sec_Communications      0.0024


,model,region,auc_mean,auc_std,f1_mean,f1_std,pr_auc_mean,pr_auc_std
0,LogisticRegression,US,0.8187,0.0423,0.5552,0.0610,0.6429,0.0752
1,DecisionTree,US,0.7118,0.0678,0.5113,0.0807,0.5112,0.0931
2,RandomForest,US,0.8037,0.0524,0.5507,0.0866,0.6310,0.0798
3,LogisticRegression,Europe,0.7472,0.0970,0.2219,0.0638,0.3199,0.1367
4,DecisionTree,Europe,0.6422,0.0904,0.1964,0.0748,0.1455,0.0687
5,RandomForest,Europe,0.7602,0.0772,0.2455,0.1528,0.3096,0.1207


## European firms without Compustat data

Firms dropped by the Compustat Global requirement, compared with the included firms.

In [5]:
eu_company["sector"] = eu_company["gics_sector_code"].apply(to_sector)
eu_company["announced"] = eu_company["all_sec_ids"].apply(lambda ids: int(any(i in adopters["EU"] for i in ids)))
eu_company["country"] = eu_company["isin"].astype(str).str[:2]
inc = eu_company["infocode"].isin(eu_df["infocode"])
groups = [("Excluded (no Compustat match)", eu_company[~inc]), ("Included", eu_company[inc])]
comparison = pd.DataFrame([dict(group=g, n_firms=len(d), announcement_rate_pct=(100 * d["announced"].mean()),
                                top_sector=d["sector"].value_counts().idxmax(),
                                top_country=d["country"].value_counts().idxmax()) for g, d in groups])
comparison.to_csv(TABLES / "eu_exclusion_comparison.csv", index=False)
for col in ("sector", "country"):
    dist = pd.concat([d[col].value_counts(normalize=True).rename(f"{name}_pct") * 100
                      for name, (_, d) in zip(("excluded", "included"), groups)], axis=1).fillna(0)
    dist.to_csv(TABLES / f"eu_exclusion_{col}_dist.csv")
comparison

,group,n_firms,announcement_rate_pct,top_sector,top_country
0,Excluded (no Compustat match),275,3.2727,Financials,GB
1,Included,619,6.4620,Industrials,GB
